# Shopee Image Moderation — NSFW + QR/Barcode

Detect **NSFW** (khỏa thân, bikini, đồ bó gợi cảm) + **QR/Barcode**. Violence / logo: implement sau.

- **NSFW**: [`LukeJacob2023/nsfw-image-detector`](https://huggingface.co/LukeJacob2023/nsfw-image-detector) — ViT 5 lớp `drawings/hentai/neutral/porn/sexy`, score = `sexy+porn+hentai`. Public, nhẹ ~340MB, chạy CPU tốt.
- **QR/Barcode**: `pyzbar` (decode trực tiếp, không AI) — có mã đọc được → vi phạm.

**Chạy trên server/local:** notebook tự tạo `.venv` trong folder project, cài đặt + cache model vào đó — KHÔNG đụng Python hệ thống / ổ C.

Thứ tự: chạy **cell Bootstrap** → đổi kernel sang *Python (moderation)* → chạy các cell còn lại. Đặt ảnh test vào `sample/`.

## 0. Bootstrap — tạo venv trong folder & cài deps (chạy 1 lần, kernel nào cũng được)

In [ ]:
import os, sys, subprocess, pathlib

PROJ = pathlib.Path.cwd()
if PROJ.name == "notebooks":
    PROJ = PROJ.parent                       # chạy được dù mở từ notebooks/ hay root
VENV = PROJ / ".venv"
PYEXE = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

if not PYEXE.exists():
    print("Tạo venv:", VENV)
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)

# OCR = RapidOCR (ONNXRuntime) thay PaddleOCR: nhanh hơn trên CPU, không dính bug oneDNN,
# không cần paddlepaddle (~2.5GB).
REQ = ["transformers>=4.50.0", "accelerate", "pillow<12", "pandas", "requests",
       "ipykernel", "pyzbar", "numpy", "rapidocr", "onnxruntime"]
print("Cài deps vào", VENV, "(lần đầu hơi lâu)...")
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", *REQ], check=True)
# torch: bản CPU cho nhẹ (~500MB). Server có GPU mới + driver CUDA 12.x thì đổi sang cu126/cu128.
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", "torch",
                "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([str(PYEXE), "-m", "ipykernel", "install", "--user",
                "--name", "moderation", "--display-name", "Python (moderation)"], check=True)
print("\nXONG. Đổi kernel: Kernel > Change Kernel > 'Python (moderation)', rồi chạy từ cell 1.")
# Lưu ý:
#   - pyzbar Windows: wheel kèm DLL zbar. Linux: sudo apt-get install -y libzbar0
#   - rapidocr: lần đầu chạy tự tải model ONNX (~vài chục MB).
#   - Nếu trước đó đã cài paddle, có thể gỡ cho nhẹ: pip uninstall paddlepaddle paddleocr

## 1. Cấu hình (chạy trong kernel *Python (moderation)*)

Đặt cache model vào folder project trước khi import transformers.

In [ ]:
import os, pathlib
PROJ = pathlib.Path.cwd()
if PROJ.name == "notebooks":
    PROJ = PROJ.parent
os.environ["HF_HOME"] = str(PROJ / ".hf_cache")   # weights tải về đây, không vào ~/.cache (C:)
IMAGE_DIR = str(PROJ / "sample")                  # thư mục ảnh test
print("HF_HOME  :", os.environ["HF_HOME"])
print("IMAGE_DIR:", IMAGE_DIR)

## 2. Load model NSFW (5 lớp)

In [ ]:
import torch
from transformers import (pipeline, AutoModelForImageClassification,
                          AutoImageProcessor, ViTImageProcessor)

NSFW_MODEL = "LukeJacob2023/nsfw-image-detector"
_device = 0 if torch.cuda.is_available() else -1

_mdl = AutoModelForImageClassification.from_pretrained(NSFW_MODEL)
try:
    _ip = AutoImageProcessor.from_pretrained(NSFW_MODEL)
except Exception:
    _ip = ViTImageProcessor.from_pretrained(NSFW_MODEL)  # repo thiếu image_processor_type
nsfw_clf = pipeline("image-classification", model=_mdl, image_processor=_ip, device=_device)
print("NSFW labels:", list(_mdl.config.id2label.values()), "| device:", _device)

## 3. Hàm moderation

In [ ]:
from PIL import Image
from pyzbar.pyzbar import decode as zbar_decode
import requests, io

_HEADERS = {"User-Agent": "Mozilla/5.0"}

def load_image(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if str(src).startswith("http"):
        r = requests.get(src, headers=_HEADERS, timeout=30); r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(src).convert("RGB")

# ---------- QR / Barcode (pyzbar) ----------
CODE_SCALES = (1.0, 1.5, 2.0)
def detect_codes(img):
    g = img.convert("L")
    found = {}
    for s in CODE_SCALES:
        im = g if s == 1.0 else g.resize((round(g.width * s), round(g.height * s)))
        for d in zbar_decode(im):
            found[(d.type, bytes(d.data))] = None
        if found:
            break
    return [{"type": t, "data": data.decode("utf-8", "replace")} for (t, data) in found]

# ---------- NSFW ----------
UNSAFE_LABELS = {"porn", "hentai", "sexy"}
ALL_LABELS = ["neutral", "drawings", "sexy", "porn", "hentai"]
NSFW_THR = 0.5
def nsfw_breakdown(img):
    return {p["label"].lower(): float(p["score"]) for p in nsfw_clf(img)}

# ---------- Tổng hợp: QR -> NSFW -> OCR, theo thứ tự CHI PHÍ, dừng sớm ----------
# OCR (chậm nhất ~1s) chạy CUỐI: frame bị QR/NSFW loại trước thì không phải OCR.
# OCR chỉ chạy nếu đã load engine (cell 6) -> moderate vẫn dùng được khi chưa bật OCR.
def moderate(src):
    img = load_image(src)

    # 1) QR/barcode — rẻ nhất, chắc chắn
    codes = detect_codes(img)
    if codes:
        return {"codes": codes, "nsfw": None, "verdict": "REJECT", "violations": "qr_barcode"}

    # 2) NSFW
    nb = nsfw_breakdown(img)
    nsfw = sum(nb.get(k, 0.0) for k in UNSAFE_LABELS)
    base = {**{k: nb.get(k, 0.0) for k in ALL_LABELS}, "codes": [], "nsfw": nsfw}
    if nsfw >= NSFW_THR:
        return {**base, "verdict": "REJECT", "violations": "nsfw"}

    # 3) OCR — chậm nhất, chạy cuối (nếu đã load OCR ở cell 6)
    if "extract_text" in globals() and "text_violations" in globals():
        hits = text_violations(extract_text(img))
        if hits:
            return {**base, "verdict": "REJECT", "violations": ",".join(hits)}

    return {**base, "verdict": "ACCEPT", "violations": ""}

## 4. Test 1 ảnh

In [ ]:
TEST_IMAGE = os.path.join(IMAGE_DIR, "sample-caught-01.jpg")   # đổi path hoặc URL
r = moderate(TEST_IMAGE)
print(r["verdict"], "|", r["violations"] or "-")
print("nsfw =", None if r["nsfw"] is None else round(r["nsfw"], 3), "| codes =", r["codes"])

## 5. Quét cả folder ảnh

In đủ điểm 5 lớp + tổng `nsfw` + verdict. Xem cột `sexy`/`neutral` để hiểu lý do và dò
threshold (nên có cả ảnh vi phạm LẪN ảnh sạch trong folder).

In [ ]:
import pandas as pd, glob, time

def _r(v):  # round an toàn (None -> None)
    return None if v is None else round(v, 3)

paths = sorted(p for e in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp")
               for p in glob.glob(os.path.join(IMAGE_DIR, e)))
print(f"Quét {len(paths)} ảnh trong {IMAGE_DIR}\n")

rows, t0 = [], time.time()
for i, p in enumerate(paths, 1):
    r = moderate(p)
    row = {"file": os.path.basename(p)}
    row.update({k: _r(r.get(k)) for k in ALL_LABELS})   # None nếu bị mã loại sớm
    row["nsfw"] = _r(r["nsfw"])
    row["code_types"] = ",".join(sorted({c["type"] for c in r["codes"]}))
    row["verdict"] = r["verdict"]
    row["violations"] = r["violations"]
    rows.append(row)
    print(f"[{i}/{len(paths)}] {row['file']:40} {row['verdict']:7} {row['violations']}")

df = pd.DataFrame(rows).sort_values(["verdict", "nsfw"], ascending=[True, False], na_position="first")
print(f"\nXong {len(paths)} ảnh / {time.time()-t0:.1f}s | REJECT: {(df['verdict']=='REJECT').sum()}")
df

## 6. OCR — đọc chữ + bắt keyword (TikTok / đối thủ / URL)

**RapidOCR** (model PP-OCR chạy qua ONNXRuntime) đọc chữ trên frame — nhanh hơn PaddleOCR
trên CPU, không dính bug oneDNN. Rồi lớp khớp keyword (chuẩn hoá chống né tránh `T1kT0k`,
`T.i.k.T.o.k`) gắn nhãn vi phạm. **Đang TÁCH RIÊNG để đo độ chính xác** — chưa cắm vào
`moderate()`. Brand/URL đều là ASCII nên model mặc định (ch+en) đọc tốt; tiếng Việt có thể
mất dấu nhưng không ảnh hưởng việc bắt keyword. Lần đầu chạy tự tải model ONNX.

In [ ]:
import os, glob, time, numpy as np

# RapidOCR: model PP-OCR chạy qua ONNXRuntime -> nhanh trên CPU, không bug oneDNN.
try:
    from rapidocr import RapidOCR            # rapidocr 2.x/3.x (API mới)
    _RAPID_NEW = True
except ImportError:
    from rapidocr_onnxruntime import RapidOCR   # rapidocr 1.x (API cũ)
    _RAPID_NEW = False

ocr_engine = RapidOCR()
print("RapidOCR API:", "new" if _RAPID_NEW else "old")

def _parse_rapid(res):
    """Chuẩn hoá output RapidOCR (khác theo version) -> list (text, conf)."""
    # API mới: object có .txts/.scores (hoặc .rec_texts/.rec_scores)
    for tattr, sattr in (("txts", "scores"), ("rec_texts", "rec_scores")):
        txts = getattr(res, tattr, None)
        if txts is not None:
            scs = getattr(res, sattr, None) or [1.0] * len(txts)
            return [(t, float(s)) for t, s in zip(txts, scs)]
    # API cũ: (result, elapse), result = [[box, text, score], ...]
    result = res[0] if isinstance(res, tuple) else res
    out = []
    for item in (result or []):
        try:
            _, t, s = item
            out.append((t, float(s)))
        except Exception:
            pass
    return out

def extract_text(img, min_conf=0.5):
    """Trả về list (text, conf) đọc được từ ảnh (đã lọc theo min_conf)."""
    arr = np.array(img.convert("RGB"))
    return [(t, s) for (t, s) in _parse_rapid(ocr_engine(arr)) if s >= min_conf]

# Probe + đo thời gian (in RAW type để chẩn đoán nếu parse sai)
_first = next(iter(sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))), None)
print("Test:", _first)
if _first:
    _im = load_image(_first)
    _raw = ocr_engine(np.array(_im.convert("RGB")))
    print("RAW type:", type(_raw))
    extract_text(_im)                                  # warmup
    _t = time.time(); _txt = extract_text(_im)
    print("Text:", _txt, f"({time.time()-_t:.2f}s/ảnh)")

In [ ]:
import re, unicodedata, glob, time
import pandas as pd

# Chuẩn hoá chống né tránh: leet (0->o,1->i...), bỏ space/dấu -> bắt 'T.i.k.T.o.k', chữ tách rời.
_LEET = str.maketrans({"0":"o","1":"i","3":"e","4":"a","5":"s","7":"t","@":"a","!":"i","$":"s","|":"i"})
def _norm(s):
    s = unicodedata.normalize("NFKC", s).lower().translate(_LEET)
    return re.sub(r"[^a-z0-9]", "", s)

# keyword latin -> nhãn. Token ngắn (tiki/temu/zalo) dễ false-positive -> theo dõi & tune.
BANNED = {
    "tiktok": "tiktok", "tiktokshop": "tiktok", "douyin": "tiktok",
    "lazada": "competitor", "tiki": "competitor", "sendo": "competitor", "taobao": "competitor",
    "temu": "competitor", "aliexpress": "competitor", "1688": "competitor", "shein": "competitor",
    "facebook": "competitor", "instagram": "competitor", "youtube": "competitor",
    "telegram": "competitor", "zalo": "competitor", "wechat": "competitor", "kuaishou": "competitor",
}
_BANNED_NORM = {kn: v for k, v in BANNED.items() if (kn := _norm(k)) and len(kn) >= 4}  # bỏ key rỗng/ngắn
_RAW_KEYWORDS = {"抖音": "tiktok"}            # chữ Hán khớp trên text gốc (vì _norm xoá hết)
_URL_RE = re.compile(r"(https?://|www\.|\.com|\.vn\b|@[a-z0-9_.]{3,})", re.I)

def text_violations(texts):
    hits = {}
    raw = " ".join(t for t, _ in texts)
    blob = _norm(raw)
    for kn, label in _BANNED_NORM.items():
        if kn in blob:
            hits.setdefault(label, []).append(kn)
    for kw, label in _RAW_KEYWORDS.items():
        if kw in raw:
            hits.setdefault(label, []).append(kw)
    for t, _ in texts:
        if _URL_RE.search(t):
            hits.setdefault("url", []).append(t.strip())
    return hits

# --- Cache OCR: chạy OCR 1 lần/ảnh, lưu lại. Tune keyword sau đó KHÔNG phải OCR lại. ---
# Muốn ép OCR lại (vd đổi config OCR): chạy  OCR_CACHE.clear()
try:
    OCR_CACHE
except NameError:
    OCR_CACHE = {}

paths = sorted(p for e in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp")
               for p in glob.glob(os.path.join(IMAGE_DIR, e)))
t0 = time.time()
for p in paths:
    if p not in OCR_CACHE:
        OCR_CACHE[p] = extract_text(load_image(p))
print(f"OCR xong ({time.time()-t0:.1f}s, {len(OCR_CACHE)} ảnh trong cache)\n")

# --- Matching (rẻ, chạy lại tức thì khi tune keyword) ---
rows = []
for p in paths:
    texts = OCR_CACHE[p]
    hits = text_violations(texts)
    rows.append({"file": os.path.basename(p),
                 "text": (" | ".join(t for t, _ in texts))[:70],
                 "labels": ",".join(hits),
                 "evidence": str({k: v[:3] for k, v in hits.items()})})
    print(f"{os.path.basename(p):45} {','.join(hits) or '-'}")

df_ocr = pd.DataFrame(rows)
print(f"\nCó nhãn: {(df_ocr['labels']!='').sum()}/{len(df_ocr)}")
df_ocr